# Расчёт проточной части газовой турбины
**Михальцев В.Е., Моляков В.Д. «Теория и проектирование газовой турбины», ч. 1**
(МГТУ им. Баумана, 2006) — гл. 4–9, алгоритм расчёта проточной части (п. 1–153, стр. 84–93).

**Расчётная точка цикла: π*к = 11** (зафиксирована в `Turbinis-Cycle`), отбор на охлаждение дисков и корпуса g_охл = 1,5 % (как у TPE331).
Проточная часть: **z = 4, сопловая лопатка 1-й ступени l_c1 > 20 мм при полном подводе.** 1-я ступень лёгкая (π ≈ 1,2) на малом диаметре
(D₁ = 0,65·D_т, α₁ = 11°, профилирование α₁ = const), затем 3 ступени с π ≈ 2,06 (запирание при 2,4) на D_т; выход из каждой ступени осевой (α₂ = 90°);
u_т = 350 м/с (п. 6.2 цикла, σ_л ≈ 239 МПа); повышенная скорость за турбиной (~160 м/с) уменьшает A_т и поднимает n - это и позволяет взять малый D₁.
Параметры лопаточного аппарата - в ячейке «Параметры, которые в книге выбираются»; предыдущая версия расчёта - в папке `old/`.

Исходные данные берутся **напрямую из `Turbinis-Cycle`**: при запуске этого блокнота
`cycle_link.load_cycle()` выполняет блокнот цикла в отдельном пространстве имён и забирает
оттуда параметры выбранной расчётной точки. Изменили что-то в цикле — перезапустите Calc,
и всё пересчитается.

Ход расчёта:
1. Исходные данные из цикла и «выбираемые» параметры книги.
2. Частота вращения и размеры (гл. 6, п. 1–12, 26–34): почему одной ступени мало.
3. Многоступенчатая турбина (п. 45–69) и расчёт ступеней (п. 70–145).
4. Параметры турбины в целом (п. 146–153) и сверка с циклом.
5. Профилирование лопаток по радиусу (гл. 7).
6. Решётки профилей: шаг, хорда, число лопаток (гл. 8).
7. Сводка и графики.

In [ ]:
%matplotlib inline
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

for _p in (Path.cwd(), Path.cwd() / "Turbinis-Calc"):
    if (_p / "cycle_link.py").exists():
        sys.path.insert(0, str(_p))
from cycle_link import load_cycle

Ц = load_cycle()  # namespace блокнота Turbinis-Cycle (он не меняется)

## 1. Исходные данные из цикла
Давление и температура перед турбиной, расход газа, давление за турбиной, механический КПД и
свойства газа (c_p, k, α) — те же функции, что использует цикл.

In [ ]:
R = Ц["R"]
calc_cp = Ц["calc_cp_kerosene"]  # c_p(T, α), Дж/(кг·К)
calc_k = Ц["calc_k_kerosene"]  # k(T, α)
р = Ц["р"]  # результат calc_tvd в выбранной точке цикла

α_г = float(р["α"])  # коэффициент избытка воздуха
T_г_полн = float(Ц["T_г_полн"])  # T*г, К
π_к = float(Ц["π_к_выб"])
p_г_полн = Ц["σ_г"] * Ц["σ_вх"] * Ц["p_а_полн"] * π_к  # p*г = σ_г σ_вх p*в π*к
p_т = Ц["π_с"] * Ц["p_а"]  # статическое давление за турбиной, Па
G_в = float(Ц["G_в_выб"])
g_т = float(р["g_т"])  # G_т/G_в (1+g_m)(1-g_ут-g_охл)
g_охл = float(Ц["g_охл"])  # отбор на охлаждение дисков и корпуса (цикл, как у TPE331)
g_воз = float(Ц["g_воз"])  # часть охлаждающего воздуха, возвращаемая в проточную часть за 1-й ступенью
G_c1 = G_в * g_т  # расход через СА 1-й ступени, кг/с
G_л_ост = G_в * (g_т + g_воз)  # расход после возврата охлаждающего воздуха, кг/с
η_м = float(Ц["η_м_тур"])
# Окружная скорость: при σ_л = const u_т = π/60·sqrt(σ_л/(π k_n l_т/D_т)) от площади A_т НЕ зависит.
# σ_л = 160 МПа (оценка 6.1 цикла) даёт u_т ≈ 286 м/с - ниже рекомендуемого для транспортных и компактных ГТД (250...360 и 300...400 м/с, стр. 59);
# поэтому берём вариант 6.2 цикла: u_т = 350 м/с -> σ_л ≈ 239 МПа (лопатки последней ступени холодные, T ≈ 750 К).
u_т_цикл = float(Ц["u_т_62"])  # м/с
σ_л_доп = float(Ц["σ_л_62"])  # МПа
K_n = float(Ц["K_n"])
l_т_D_т = float(Ц["l_т_to_D_т"])
c_т_а_цикл = float(Ц["c_т"])  # скорость за турбиной, принятая в цикле (M_т)
n_н = Ц["n_н"]  # частота вала нагрузки, если задана
N_т_цикл = G_в * р["L_т"] * (g_т + g_воз) * η_м  # мощность турбины в цикле, Вт
η_т_полн_цикл = float(р["η_т"])

исх = pd.Series(
    {
        "T*г, К": T_г_полн,
        "p*г, кПа": p_г_полн / 1e3,
        "p_т (за турбиной), кПа": p_т / 1e3,
        "π_т = p*г/p_т": p_г_полн / p_т,
        "G_c1 (через СА 1-й ступени), кг/с": G_c1,
        "G после возврата охл. воздуха, кг/с": G_л_ост,
        "g_охл (диски, корпус), %": g_охл * 100,
        "g_воз (возврат за 1-й ступенью), %": g_воз * 100,
        "α (избыток воздуха)": α_г,
        "η_м": η_м,
        "N_т по циклу, кВт": N_т_цикл / 1e3,
        "η*т по циклу (принят)": η_т_полн_цикл,
        "c_т по циклу, м/с": c_т_а_цикл,
        "u_т (цикл, п. 6.2), м/с": u_т_цикл,
        "σ_л доп, МПа": σ_л_доп,
        "l_т/D_т": l_т_D_т,
        "n_н (вал нагрузки)": n_н,
    }
)
print(f"Расчётная точка цикла: «{Ц['выбор']}», π*к = {π_к:.2f}")
исх.to_frame("значение")

## 2. Параметры, которые в книге «выбираются»
Значения — из рекомендаций книги (середины диапазонов); то, что есть в цикле, взято из цикла.
Все допущения собраны здесь — меняйте только эту ячейку.

In [ ]:
# --- коэффициенты скорости (стр. 34-37) ---
φ = 0.97  # СА: 0,95...0,97 (стр. 34) - верхняя граница; книга отмечает, что при α1 < 17° и l < 40...50 мм φ ниже - оценка оптимистичная
ψ = 0.96  # РК: типично 0,92...0,96 (растёт с ρ и при малом угле поворота потока, стр. 37); при 0,96/0,94 η*т ≈ 0,87
# --- углы ---
α1_цель = [11.0, 12.0, 12.0, 18.0]  # град, по ступеням: 1-я - наименьший (длинная лопатка), последняя - больше (выше c_т -> меньше A_т -> больше n); угол выхода из СА (у малых турбин 11-14°, стр. 34: меньше α1 - длиннее лопатки и меньше потери в зазоре)
α1_мин = 11.0  # град, контроль нижней границы (малые турбины 11-14°)
α1_мин_корня = 11.0  # град: у корня >= 13...14°, для турбин малой мощности допустимо 11...12° (стр. 73)
закон_профилирования = ["α1", "Γ", "Γ", "Γ"]  # закон по радиусу для каждой ступени: "Γ" (Γ1u = const) или "α1" (α1 = const, разд. 7.3);
# у 1-й ступени l/D ~ 1/3 - при Γ = const α1 у корня < 11° и α1_ср пришлось бы поднять до ~17° (лопатка короче)
α2_диап = (89.0, 91.0)  # допуск α2 ВСЕХ ступеней: выход из каждой ступени осевой (α2 = 90°)
α2_цель = 90.0  # град
κ_ca = [0.85, 0.85, 0.85, 1.2]  # по ступеням; c2a = κ·c1a: при α2 = 90° c2 = c2a, κ < 1 снижает выходную скорость до принятой в цикле (~108 м/с)
# --- число ступеней и распределение теплоперепада (п. 51-55) ---
z_заданное = 4  # число ступеней; при l_c1 > 20 мм 1-я ступень лёгкая (π ≈ 1,2), и на 3 ступени π > 2,4 (запирание) - нужно 4
y0 = 0.539539  # оптимальное (u/c0) ступени - только для оценки z по формуле п. 54 (при заданном z u/c0 ступеней получается из π)
μ = 0.9  # коэффициент использования скорости c2 следующей ступенью
D1_к_Dт = 1.0  # D1/D_т: 1 -> постоянный средний диаметр; < 1 удлиняет лопатки 1-й ступени, но снижает u1: при 0,85 у 1-й ступени u/c0 ≈ 0,45 и α2 = 90° не получить
π_ст_макс = 2.4  # p*0/p2 ступени: выше - запирание (кризис) решёток
π_ст_цель = (1.8, 2.0)  # желательный диапазон π ступени
π_веса = [0.25, 1.0, 1.0, 1.0]  # распределение π по ступеням: ln π_i ~ вес (None - равные π); 1-я ступень легче (у неё самые короткие лопатки)
# --- степень реактивности (стр. 58): ρ_ср = f(l/D) ---
ρ_таблица = ([1 / 9, 1 / 6, 1 / 4, 1 / 3], [0.2, 0.3, 0.4, 0.5])
ρ_макс = 0.6
ρ_корня_мин = 0.05  # у корня ρ >= 0,05...0,1
# --- потери ---
δ_отн = 0.005  # относительный радиальный зазор δ_л/l (0,007...0,01 без уплотнения; соты/бандаж - меньше, стр. 12, 62)
δ_мин = 0.2e-3  # м, минимальный абсолютный зазор: сотовые вставки, после приработки <= 0,2 мм (стр. 12); при 0,3 мм η*т ниже на ~0,012
ε_парц = 1.0  # степень парциальности: полный подвод (парциальность в авиационных ГТД не применяется)
D_отн = [0.65, 1.0, 1.0, 1.0]  # средний диаметр каждой ступени D/D_т (последний = 1); None - линейно от D1_к_Dт до 1.
# Длина лопатки 1-й ступени l1 = G v1/(π D1 c1a): малый D1 и малая c1a (лёгкая ступень, α1 = 11°) при большой n
l_c1_мин = 20e-3  # м, требуемая наименьшая длина сопловой лопатки 1-й ступени
# --- меридиональное сечение (стр. 55, 61-62) ---
lb_c, lb_л = 2.25, 2.75  # удлинение (l/b_a) СА и РК - исходное (одноступенчатые: 2...2,5 и 2,5...3)
lb_мин_c, lb_мин_л = 1.0, 1.5  # наименьшее удлинение при расширении венцов (ТВД: 1...1,5 и 1,5...2)
δа_к_ba = 0.22  # осевой зазор δ_a/b_a,c = 0,20...0,25
b_a_мин = 5e-3  # м, конструктивный минимум ширины венца (оценка, не из книги)
γ_цель = 12.0  # град, желательный угол раскрытия γ = γ_н + γ_в (книга: 0...20°)
γ_макс = 25.0  # град, больше - отрыв потока на контуре (стр. 55): тогда увеличиваем z
# --- прочее ---
η_л_ожид = 0.90  # п. 5: лопаточный КПД для оценки T_т
охл_ступеней = 0  # число охлаждаемых ступеней (для толщин кромок, гл. 8): лопатки неохлаждаемые (охлаждаются только диски)


## 3. Свойства газа и функции, общие для расчёта
c_p и k — **истинные** (по книге, стр. 15 и 24), а не средние 0...T из методички цикла.


In [ ]:
def газ(T):
    """Истинные c_p и k при температуре T для продуктов сгорания с α из цикла (стр. 15, 24, рис. П1-П2).

    Функция цикла calc_cp_kerosene даёт СРЕДНЮЮ теплоёмкость c_pm в интервале 0 °C...T
    (методичка, Прил. 6). Истинная: c_p = d(c_pm·t)/dt = c_pm + t·dc_pm/dT, t = T - 273,15 °C.
    Показатель адиабаты: k = c_p/(c_p - R).
    """
    dT = 1.0
    c_pm = float(calc_cp(T, α_г))
    dc_pm = (float(calc_cp(T + dT, α_г)) - float(calc_cp(T - dT, α_г))) / (2 * dT)
    c_p = c_pm + (T - 273.15) * dc_pm
    return c_p, c_p / (c_p - R)


class НетРешения(Exception):
    pass


def ρ_табл(l_к_D):
    return float(np.interp(l_к_D, *ρ_таблица))


def cp_k_ступени(T0, H0):
    """п. 75-76: c_p и k по средней условной температуре T_у = T*0 - H0 φ²/(2 c_p)."""
    cp, k = газ(T0)
    for _ in range(50):
        T_у = T0 - H0 * φ**2 / (2 * cp)
        cp_н, k_н = газ(T_у)
        if abs(cp_н - cp) < 1e-9:
            break
        cp, k = cp_н, k_н
    return cp, k


def H0_по_p(T0, p0, p2):
    """п. 140: располагаемый теплоперепад ступени при заданном p2."""
    cp, k = газ(T0)
    H0 = 0.0
    for _ in range(50):
        H0 = cp * T0 * (1 - (p2 / p0) ** ((k - 1) / k))
        cp_н, k_н = газ(T0 - H0 * φ**2 / (2 * cp))
        if abs(cp_н - cp) < 1e-9:
            break
        cp, k = cp_н, k_н
    return H0

## 4. Частота вращения и размеры одноступенчатой турбины (гл. 6, п. 1–12, 26–34)
Сначала — первая оценка по статическому давлению за турбиной (вариант «задано p_т»).

In [ ]:
# п. 1-4
π_г = p_г_полн / p_т
cp0, k0 = газ(0.5 * (T_г_полн + T_г_полн * π_г**-0.25))  # T_ср = 0,5(T*г + T_ту), T_ту = T*г π_т^(-1/4)
H_т = cp0 * T_г_полн * (1 - π_г ** ((1 - k0) / k0))  # (1)
c0_т = math.sqrt(2 * H_т)
# п. 5-9
T_т = T_г_полн - H_т * η_л_ожид / cp0
v_т = R * T_т / p_т
A_т0 = G_л_ост * v_т / c_т_а_цикл  # c_т.а берём из цикла (M_т)
# п. 10-12
n_max0 = math.sqrt(σ_л_доп / (K_n * A_т0))  # σ_л = k_n n² A_т
n_т0 = n_max0 if n_н is None else float(n_н)
# п. 45-47 (D_т по l/D) и 26-31 (оптимальный диаметр одной ступени)
D_т0 = math.sqrt(A_т0 / (math.pi * l_т_D_т))
u_т0 = math.pi * D_т0 * n_т0 / 60
u_η = y0 * c0_т  # п. 28-29
D_η = 60 * u_η / (math.pi * n_т0)  # п. 30
l_η = A_т0 / (math.pi * D_η)  # п. 31

print(f"π_т(стат.) = {π_г:.2f}, k = {k0:.4f}, c_p = {cp0:.0f} Дж/(кг·К), H_т = {H_т/1e3:.1f} кДж/кг")
print(f"T_т = {T_т:.0f} К, v_т = {v_т:.3f} м³/кг, A_т = {A_т0*1e4:.1f} см², n_т max = {n_max0:.0f} об/мин")
print(f"c0 = {c0_т:.0f} м/с; одна ступень: u_η = {u_η:.0f} м/с (при u_т = {u_т0:.0f} м/с), "
      f"D_η = {D_η*1e3:.0f} мм против D_т = {D_т0*1e3:.0f} мм, l_η = {l_η*1e3:.1f} мм")
if u_η > 400 or D_η > 1.5 * D_т0:
    print("ВЫВОД (замечание 1, стр. 94; п. 29, 32): одноступенчатая турбина нецелесообразна — проектируем многоступенчатую.")

## 5. Ступень турбины (п. 70–129)
`ступень()` — полный расчёт одной ступени на среднем радиусе: треугольники скоростей,
параметры газа, потери в СА и РК, в радиальном зазоре, на трение и вентиляцию, КПД.
Давление за ступенью следует из располагаемого теплоперепада: p₂ = p*₀ (1 − H₀/(c_p T*₀))^(k/(k−1)).

In [ ]:
def ступень(T0, p0, H0, ρ, D_c, D_л, n, G_c, G_л, α1_град, κ=None):
    cp, k = cp_k_ступени(T0, H0)  # п. 75-76
    # --- сопловой аппарат (п. 36-42, 57-62) ---
    Hc = H0 * (1 - ρ)
    c1 = φ * math.sqrt(2 * Hc)
    T1 = T0 - Hc * φ**2 / cp
    T1_ад = T0 - Hc / cp
    p1 = p0 * (T1_ад / T0) ** (k / (k - 1))
    v1 = R * T1 / p1
    # п. 63: l_c = G v1 / (π D sinα1 c1) при заданном α1 (раскрытие контура регулируется шириной венцов, разд. 9)
    l_c = G_c * v1 / (math.pi * D_c * math.sin(math.radians(α1_град)) * c1)
    c1a = G_c * v1 / (math.pi * D_c * l_c)
    if c1a >= c1:
        raise НетРешения("c1a >= c1")
    sin_α1 = c1a / c1
    α1 = math.asin(sin_α1)
    c1u = c1 * math.cos(α1)
    # --- вход в РК (п. 84-88) ---
    u1 = math.pi * D_c * n / 60
    u2 = math.pi * D_л * n / 60
    w1 = math.sqrt(c1**2 + u1**2 - 2 * c1 * u1 * math.cos(α1))
    β1 = math.atan2(c1a, c1u - u1)
    Hл = ρ * H0 * T1 / T1_ад  # п. 86
    # --- выход из РК (п. 89-100) ---
    w2 = ψ * math.sqrt(w1**2 + 2 * Hл + u2**2 - u1**2)
    T2 = T1 - (w2**2 - w1**2) / (2 * cp) + (u2**2 - u1**2) / (2 * cp)
    T2_ад = T1 - Hл / cp
    p2 = p1 * (T2_ад / T1) ** (k / (k - 1))
    v2 = R * T2 / p2
    c2a = (κ_ca if κ is None else κ) * c1a
    if w2 <= c2a:
        raise НетРешения("w2 <= c2a")
    c2u = math.sqrt(w2**2 - c2a**2) - u2  # c2u = w2 cosβ2 - u2
    β2 = math.atan2(c2a, c2u + u2)
    α2 = math.degrees(math.atan2(c2a, c2u))
    c2 = math.hypot(c2a, c2u)
    l_л = G_л * v2 / (c2a * math.pi * D_л)  # п. 95-96
    p2_полн = p2 * (1 + c2**2 / (2 * cp * T2)) ** (k / (k - 1))
    # --- работа и КПД на окружности (п. 101-111) ---
    L_u = c1u * u1 + c2u * u2
    η_u = L_u / H0
    ξ_c = (φ**-2 - 1) * c1**2 / 2 / H0 * T2_ад / T1  # ξ'_c
    ξ_л = (ψ**-2 - 1) * w2**2 / 2 / H0
    ξ_в = c2**2 / 2 / H0
    η_u2 = 1 - ξ_c - ξ_л - ξ_в
    # --- радиальный зазор (п. 114-118) ---
    D_ср = 0.5 * (D_c + D_л)
    δ_л = max(δ_отн * l_л, δ_мин)
    h_з = 1.37 * (1 + 1.6 * ρ) * (1 + l_л / D_ср) * (δ_л / l_л) * L_u
    ξ_з = h_з / H0
    η_т1 = η_u - ξ_з
    η_л1 = η_т1 + ξ_в
    # --- трение и вентиляция, формула Стодола (п. 119-123) ---
    ρ_г = 0.5 * (1 / v1 + 1 / v2)
    u_ср = 0.5 * (u1 + u2)
    N_тв = (1.07 * D_ср**2 + 61 * (1 - ε_парц) * D_ср * l_л) * (u_ср / 100) ** 3 * ρ_г * 1e3
    h_тв = N_тв / G_c
    ξ_тв = h_тв / H0
    η_т = η_т1 - ξ_тв
    η_л = η_л1 - ξ_тв
    L_т = H0 * η_т  # п. 124
    # --- параметры за ступенью (п. 125-129) ---
    T_ст = T2 + (h_з + h_тв) / cp
    T_ст_п = T_ст + c2**2 / (2 * cp)
    p_ст_п = p2 * (T_ст_п / T_ст) ** (k / (k - 1))
    H0_п = cp * T0 * (1 - (p_ст_п / p0) ** ((k - 1) / k))
    return dict(
        T0=T0, p0=p0, H0=H0, ρ=ρ, cp=cp, k=k, Hc=Hc, Hл=Hл,
        c1=c1, c1a=c1a, c1u=c1u, α1=math.degrees(α1), T1=T1, T1_ад=T1_ад, p1=p1, v1=v1, l_c=l_c,
        u1=u1, u2=u2, w1=w1, β1=math.degrees(β1), w2=w2, β2=math.degrees(β2), T2=T2, p2=p2, v2=v2,
        c2a=c2a, c2u=c2u, c2=c2, α2=α2, l_л=l_л,
        L_u=L_u, η_u=η_u, η_u2=η_u2, ξ_c=ξ_c, ξ_л=ξ_л, ξ_в=ξ_в, ξ_з=ξ_з, ξ_тв=ξ_тв, h_з=h_з, h_тв=h_тв,
        η_т=η_т, η_л=η_л, L_т=L_т, T_ст=T_ст, T_ст_п=T_ст_п, p_ст_п=p_ст_п, H0_п=H0_п, η_т_п=L_т / H0_п, π_ст=p0 / p2, π_ст_полн=p0 / p2_полн,
        D_c=D_c, D_л=D_л, G_c=G_c, G_л=G_л, δ_л=δ_л,
    )  # fmt: skip


def ρ_корня(r):
    """Реактивность у корня СА при профилировании Γ = const (формулы (71)-(73), стр. 70-71)."""
    k, cp, T0, p0, H0, p2 = r["k"], r["cp"], r["T0"], r["p0"], r["H0"], r["p2"]
    r_c = r["D_c"] / 2
    r1 = r_c - 0.5 * r["l_c"]
    M1u = r["c1u"] / math.sqrt(k * R * r["T1_ад"])
    p1 = r["p1"] * (1 + (k - 1) / 2 * M1u**2 * (1 - (r_c / r1) ** 2)) ** (k / (k - 1))
    T1ад = T0 * (p1 / p0) ** ((k - 1) / k)
    return cp * T1ад / H0 * (1 - (p2 / p1) ** ((k - 1) / k))


def _ступень_авто(T0, p0, H0, D_c, D_л, n, G_c, G_л, диап, α1_град, проверка_корня=True, α2_цель=None, κ=None):
    """ρ по l/D (таблица стр. 58); если α2 вне допуска — меняем ρ и пересчитываем (п. 99)."""

    def расчёт(ρ):
        return ступень(T0, p0, H0, ρ, D_c, D_л, n, G_c, G_л, α1_град, κ)

    def α2_от(ρ):
        try:
            return расчёт(ρ)["α2"]
        except НетРешения:
            return 180.0

    ρ = ρ_табл(0.1)
    for _ in range(6):
        if α2_от(ρ) >= 180.0:
            break
        ρ_н = ρ_табл(расчёт(ρ)["l_л"] / D_л)
        if abs(ρ_н - ρ) < 0.003:
            break
        ρ = ρ_н
    if not диап[0] <= α2_от(ρ) <= диап[1]:
        цель = α2_цель if α2_цель is not None else диап[1] - 2.0  # α2 убывает с ростом ρ
        lo, hi = 0.02, ρ_макс
        for _ in range(40):
            mid = 0.5 * (lo + hi)
            lo, hi = (mid, hi) if α2_от(mid) > цель else (lo, mid)
        ρ = 0.5 * (lo + hi)
    if проверка_корня:
        # стр. 58: у корня лопатки ρ >= 0,05...0,1. Нулевая закрутка (α2 ~ 88°) у длинных лопаток даёт у корня ρ ~ 0 -
        # повышаем ρ_ср, пока не выполнится условие у корня (допуская остаточную закрутку, α2 >= диап[0])
        def корень(ρ_):
            try:
                r_ = расчёт(ρ_)
            except НетРешения:
                return None
            return ρ_корня(r_) if r_["α2"] >= диап[0] else None

        if (корень(ρ) or -1.0) < ρ_корня_мин:
            lo, hi = ρ, ρ_макс
            for _ in range(40):
                mid = 0.5 * (lo + hi)
                f = корень(mid)
                if f is not None and f < ρ_корня_мин:
                    lo = mid
                else:
                    hi = mid
            ρ = hi
    r = расчёт(ρ)
    r["ρ_табл"] = ρ_табл(r["l_л"] / D_л)
    return r


def ступень_авто(T0, p0, H0, D_c, D_л, n, G_c, G_л, диап, α1_град, проверка_корня=True, α2_цель=None, κ=None, закон="Γ"):
    """Как _ступень_авто, но α1 повышается, если при профилировании Γ = const он у корня падает ниже α1_мин_корня.

    У корня tg α1 = tg α1_ср · r_корня / r_ср (c1u·r = const), поэтому у длинных лопаток нужен больший α1_ср.
    """
    if закон != "Γ":  # α1 = const по радиусу: у корня тот же α1, повышать α1_ср не нужно
        return _ступень_авто(T0, p0, H0, D_c, D_л, n, G_c, G_л, диап, α1_град, проверка_корня=проверка_корня, α2_цель=α2_цель, κ=κ)
    α1 = α1_град
    for _ in range(8):
        r = _ступень_авто(T0, p0, H0, D_c, D_л, n, G_c, G_л, диап, α1, проверка_корня=проверка_корня, α2_цель=α2_цель, κ=κ)
        r_ср, r_кор = r["D_c"] / 2, r["D_c"] / 2 - r["l_c"] / 2
        α1_нужен = math.degrees(math.atan(math.tan(math.radians(α1_мин_корня)) * r_ср / r_кор))
        α1_нужен += 0.15  # запас на сходимость
        if r["α1"] >= α1_нужен:
            break
        α1 = α1 + max(α1_нужен - r["α1"], 0.1)
    return r


## 6. Многоступенчатая турбина (п. 45–69, 130–145)
* Средний диаметр последней ступени — по A_т и l/D (п. 45–46), окружная скорость — по n_т.
  При σ_л = const окружная скорость u_т = π/60·√(σ_л/(π k_n l_т/D_т)) от площади не зависит, поэтому её задаёт выбор σ_л (u_т = 350 м/с из п. 6.2 цикла).
* Число ступеней по п. 54 (при оптимальном u/c₀) печатается для сравнения; принимаем **z = 3** (`z_заданное`), даже если формула даёт 3,6…3,8.
* Теплоперепад ступеней задаётся через **π = p*₀/p₂ ступени** (H₀ по п. 140): ln π_i ∝ `π_веса`, последняя ступень — по давлению за турбиной
  p₂z = p_т (п. 138–141); веса подобраны так, чтобы π ≤ 2,4 (запирание) и были близки к 1,8…2,0. Подток энергии c₂ предыдущей ступени учтён (п. 130–134).
* У каждой ступени α₂ = 90° (осевой выход): степень реактивности ρ подбирается под это условие (п. 99).
* Расход воздуха G_в подбирается так, чтобы мощность на валу (N_т − N_к)·η_р равнялась заданной N_зад.
* **Длинная лопатка 1-й ступени (l_c1 > 20 мм) без парциальности.** По неразрывности l₁ = G·v₁/(π·D₁·c₁a), а D₁ = 60·u₁/(π·n):
  лопатка тем длиннее, чем меньше c₁a и u₁ (лёгкая 1-я ступень, α₁ = 11°, D₁ = 0,65·D_т) и чем больше n. n ограничена σ_л последней ступени
  (σ_л = k_n n² A_т), поэтому скорость за турбиной повышена (α₁ = 18°, κ = 1,2 в последней ступени) - A_т меньше, n больше.
  После лёгкой 1-й ступени на 2 ступени пришлось бы π ≈ 2,9 > 2,4, поэтому z = 4. Между 1-й и 2-й ступенью - переходный канал с ростом диаметра.
* За 1-й ступенью в поток возвращается часть воздуха, охлаждавшего диски (g_воз из цикла).

Площадь A_т (п. 8–9) была выбрана по c_т.а из цикла. После расчёта ступеней фактическая
площадь за последней ступенью может отличаться, поэтому A_т, n_т max и D_т уточняются
итерацией до совпадения (σ_л при этом остаётся равным допустимому).


In [ ]:
p_кон = p_т  # давление за последней ступенью; подбирается ниже из условия N_вал = N_зад


def турбина(z, n, D1, D_т, π_ст_):
    """Ступени 1...z-1 - по заданной π = p*0/p2 = π_ст_^(w_i/w_z) (H0 по п. 140), последняя - по p2z = p_т (п. 138-141).

    α2 = 90° у всех ступеней. Подток энергии c2 предыдущей ступени учтён в p*0 (п. 130-134)."""
    D_гр = np.linspace(D1, D_т, z + 1)  # средние диаметры сечений: выход СА1, выходы РК1...РКz
    res = []
    for i in range(1, z + 1):
        D_c, D_л = float(D_гр[i - 1]), float(D_гр[i])
        if D_отн:  # средний диаметр ступени задан (СА и РК на одном диаметре, между ступенями - переходный канал)
            D_c = D_л = D_отн[i - 1] * D_т
        u = math.pi * D_c * n / 60
        if i == 1:  # п. 57-63
            T0, p0, G_c, G_л = T_г_полн, p_г_полн, G_c1, G_c1
        else:  # п. 130-134; за 1-й ступенью в поток возвращается охлаждающий воздух дисков (g_воз)
            пр = res[-1]
            T0 = пр["T_ст_п"]
            p0 = пр["p2"] * (1 + (μ * пр["c2"]) ** 2 / (2 * пр["cp"] * пр["T_ст"])) ** (пр["k"] / (пр["k"] - 1))
            G_c = G_л = G_л_ост
        if i < z:
            w = π_веса[i - 1] / π_веса[z - 1] if π_веса else 1.0
            H0 = H0_по_p(T0, p0, p0 / π_ст_**w)
        else:  # п. 138-141: p2z = p_т
            if p0 <= p_кон * 1.02:
                raise НетРешения("давление перед последней ступенью не выше p_т: z слишком велико")
            H0 = H0_по_p(T0, p0, p_кон)
        пс = lambda x: x[i - 1] if isinstance(x, (list, tuple)) else x  # параметр ступени: число или список по ступеням
        res.append(ступень_авто(T0, p0, H0, D_c, D_л, n, G_c, G_л, α2_диап, пс(α1_цель), α2_цель=α2_цель, κ=пс(κ_ca), закон=пс(закон_профилирования)))
    return res


def u_c0(s):
    """u/c0 ступени (c0 = sqrt(2 H0), H0 - с учётом подтока c2 предыдущей ступени)."""
    return s["u1"] / math.sqrt(2 * s["H0"])


def равная_нагрузка(z, n, D1, D_т, y_старт):
    """При заданном z: π ступеней в пропорции ln π_i ~ π_веса (по умолчанию равные), секущие по π последней ступени."""

    def f(y):
        try:
            ст = турбина(z, n, D1, D_т, y)
        except НетРешения:
            return None, None
        return ст[-1]["π_ст"] - y, ст

    y_a, y_b = y_старт, y_старт * 1.02
    f_a, ст = f(y_a)
    if f_a is None:
        raise НетРешения("нет решения при начальной π ступени")
    for _ in range(30):
        f_b, ст_b = f(y_b)
        if f_b is None:  # слишком большая нагрузка последней ступени - назад
            y_b = 0.5 * (y_a + y_b)
            continue
        if abs(f_b) < 1e-5:
            return y_b, ст_b
        y_a, y_b, f_a = y_b, y_b - f_b * (y_b - y_a) / (f_b - f_a), f_b
        y_b = float(np.clip(y_b, 1.05, π_ст_макс * 1.2))
    return y_b, ст_b


def решить(p_к, y_старт):
    """Полный расчёт турбины (итерация по A_т) при давлении за последней ступенью p_к."""
    global p_кон
    p_кон = p_к
    A_т = A_т0
    c2_ср = c_т_а_цикл  # п. 52: начальная оценка, затем - по результатам расчёта
    итерации = []
    y = y_старт or (p_г_полн / p_к) ** (1 / (z_заданное or 3))  # первое приближение: равные π
    for it in range(40):
        n_max = math.sqrt(σ_л_доп / (K_n * A_т))
        n_т = n_max if n_н is None else float(n_н)
        D_т = math.sqrt(A_т / (math.pi * l_т_D_т))  # п. 46
        D1 = D1_к_Dт * D_т  # п. 48
        u_т = math.pi * D_т * n_т / 60  # п. 47
        u_ср = 0.5 * (u_т + math.pi * D1 * n_т / 60)  # п. 49-50
        μc = μ * c2_ср
        z_расч = (c0_т**2 / μc**2 - 1) / (u_ср**2 / (μc**2 * y0**2) - 1)  # п. 54 (при y0 из допущений)
        z = z_заданное or max(1, math.ceil(z_расч))
        y, ступени = равная_нагрузка(z, n_т, D1, D_т, y)
        A_факт = math.pi * ступени[-1]["D_л"] * ступени[-1]["l_л"]
        c2_ср = float(np.mean([s["c2"] for s in ступени]))
        итерации.append((it, A_т, n_т, D_т, u_т, z_расч, z, y, A_факт))
        if abs(A_факт - A_т) / A_т < 1e-4:
            break
        A_т = 0.5 * (A_т + A_факт)
    return dict(A_т=A_т, n_т=n_т, D_т=D_т, D1=D1, u_т=u_т, u_ср=u_ср, A_факт=A_факт, z=z, z_расч=z_расч, π_ст=y,
                ступени=ступени, итерации=итерации)


def задать_расход(G):
    """Меняем расход воздуха: G_в, расходы через СА/РК и начальная оценка A_т пропорциональны ему."""
    global G_в, G_c1, G_л_ост, A_т0
    k = G / G_в
    G_в, G_c1, G_л_ост, A_т0 = G, G_c1 * k, G_л_ост * k, A_т0 * k


_y_тек = None


def N_на_валу(G):
    """Мощность на валу (п. 146, 153): N_в = (N_т - N_к) η_р, как в цикле; p за последней ступенью = p_т."""
    global _y_тек
    задать_расход(G)
    r = решить(p_т, _y_тек)
    _y_тек = r["π_ст"]
    ст = r["ступени"]
    g = [s["G_c"] / G_c1 for s in ст]
    N_турб = sum(gi * s["L_т"] for gi, s in zip(g, ст)) * G_c1 * η_м
    return (N_турб - G_в * float(р["L_к"])) * float(Ц["η_р"]), r


# Расход воздуха из цикла получен при цикловом η*т; у проточной части книги КПД иной, поэтому
# подбираем G_в так, чтобы чистая мощность на валу была равна заданной (давление за турбиной = p_т цикла)
N_зад = float(Ц["N_зад"])
G_в_цикл = G_в
G_a, G_b = G_в_цикл, 1.05 * G_в_цикл  # метод секущих: N_вал почти линейна по G_в
N_a, _r = N_на_валу(G_a)
for _ in range(20):
    N_b, _r = N_на_валу(G_b)
    if abs(N_b - N_зад) < 1e-5 * N_зад:
        break
    G_a, G_b, N_a = G_b, G_b + (N_зад - N_b) * (G_b - G_a) / (N_b - N_a), N_b
N_вал_подбор = N_b
p_кон = p_т
A_т, n_т, D_т, D1, u_т, u_ср, A_факт, z, z_расч, ступени, итерации = (
    _r[k] for k in ("A_т", "n_т", "D_т", "D1", "u_т", "u_ср", "A_факт", "z", "z_расч", "ступени", "итерации")
)

σ_л_факт = K_n * n_т**2 * A_факт
print(pd.DataFrame(итерации, columns=["итер.", "A_т, м²", "n_т, об/мин", "D_т, м", "u_т, м/с", "z (п.54)", "z", "π ступени", "A_2a факт., м²"]).to_string(index=False))
print(f"Подобран расход воздуха G_в = {G_в:.4f} кг/с (по циклу {G_в_цикл:.4f} кг/с, {(G_в/G_в_цикл-1)*100:+.1f} %), "
      f"p за последней ступенью = p_т = {p_т/1e3:.2f} кПа, N_вал = {N_вал_подбор/1e3:.2f} кВт")
print(f"\nz = {z} (по п. 54 при u/c0 = {y0:.3f}: z = {z_расч:.2f}); u/c0 по ступеням: " + ", ".join(f"{u_c0(s):.3f}" for s in ступени))
print(f"n_т = {n_т:.0f} об/мин, D_т = {D_т*1e3:.1f} мм, D1 = {D1*1e3:.1f} мм, u_т = {u_т:.0f} м/с, σ_л = {σ_л_факт:.0f} МПа (доп. {σ_л_доп:.0f})")
print("π ступеней p*0/p2: " + ", ".join(f"{s['π_ст']:.2f}" for s in ступени)
      + f"  (желательно {π_ст_цель[0]}...{π_ст_цель[1]}, запирание при {π_ст_макс})")


## 7. Результаты по ступеням (п. 70–145)

In [ ]:
имена = [
    ("H0", "H₀ ступени", "кДж/кг", 1e-3),
    ("π_ст", "π = p*₀/p₂", "-", 1),
    ("π_ст_полн", "π* = p*₀/p*₂", "-", 1),
    ("ρ", "ρ (реактивность)", "-", 1),
    ("c1", "c₁", "м/с", 1),
    ("α1", "α₁", "град", 1),
    ("β1", "β₁", "град", 1),
    ("w1", "w₁", "м/с", 1),
    ("w2", "w₂", "м/с", 1),
    ("β2", "β₂", "град", 1),
    ("c2", "c₂", "м/с", 1),
    ("α2", "α₂", "град", 1),
    ("u1", "u₁", "м/с", 1),
    ("T1", "T₁", "К", 1),
    ("p1", "p₁", "кПа", 1e-3),
    ("p2", "p₂", "кПа", 1e-3),
    ("T2", "T₂", "К", 1),
    ("l_c", "l_c (СА)", "мм", 1e3),
    ("l_л", "l_л (РК)", "мм", 1e3),
    ("L_u", "L_u", "кДж/кг", 1e-3),
    ("η_u", "η_u (п. 102)", "-", 1),
    ("η_u2", "η_u (п. 111)", "-", 1),
    ("ξ_c", "ξ'_c", "-", 1),
    ("ξ_л", "ξ_л", "-", 1),
    ("ξ_в", "ξ_в", "-", 1),
    ("ξ_з", "ξ_з (рад. зазор)", "-", 1),
    ("ξ_тв", "ξ_т.в", "-", 1),
    ("η_т", "η_т ступени", "-", 1),
    ("η_л", "η_л ступени", "-", 1),
    ("η_т_п", "η*_т ступени", "-", 1),
    ("L_т", "L_т", "кДж/кг", 1e-3),
]
табл_ст = pd.DataFrame(
    {f"ст. {i}": [s[к] * м for к, _, _, м in имена] for i, s in enumerate(ступени, 1)},
    index=pd.MultiIndex.from_tuples([(н, е) for _, н, е, _ in имена], names=["параметр", "ед."]),
)
табл_ст.style.format("{:.4g}")

## 8. Турбина в целом (п. 146–153) и сверка с циклом

In [ ]:
g_i = [s["G_c"] / G_c1 for s in ступени]
L_тΣ = sum(g * s["L_т"] for g, s in zip(g_i, ступени))  # п. 146
H_т_ст = H_т  # п. 147: по истинному p_т
η_т_турб = L_тΣ / H_т_ст  # п. 148
h_вz = ступени[-1]["c2"] ** 2 / 2
η_л_турб = (L_тΣ + h_вz) / H_т_ст  # п. 149
p_т_полн = ступени[-1]["p_ст_п"]  # p*_т за последней ступенью
cp_т, k_т = газ(0.5 * (T_г_полн + ступени[-1]["T_ст_п"]))
H_т_полн = cp_т * T_г_полн * (1 - (p_т_полн / p_г_полн) ** ((k_т - 1) / k_т))  # п. 150
η_т_полн = L_тΣ / H_т_полн  # п. 151
N_т = L_тΣ * G_c1 * η_м  # п. 153
# мощность на валу (в редуктор и на винт): N_в = (N_т - N_к) η_р, как в цикле
N_к = G_в * float(р["L_к"])
N_вал = (N_т - N_к) * float(Ц["η_р"])
N_вал_цикл = (N_т_цикл - G_в_цикл * float(р["L_к"])) * float(Ц["η_р"])  # при расходе цикла

сверка = pd.DataFrame(
    {
        "по книге (Calc)": [N_вал / 1e3, N_т / 1e3, G_в, η_т_полн, p_т_полн / 1e3, ступени[-1]["c2"], ступени[-1]["T_ст_п"]],
        "по циклу (Cycle)": [Ц["N_зад"] / 1e3, N_т_цикл / 1e3, G_в_цикл, η_т_полн_цикл, Ц["р"]["π_с_полн"] * Ц["p_а"] / 1e3, c_т_а_цикл, float(р["T_т"])],
    },
    index=["N на валу, кВт", "N_т, кВт", "G_в, кг/с", "η*_т", "p*_т за турбиной, кПа", "скорость за турбиной, м/с", "T*_т, К"],
)
сверка["расхождение, %"] = (сверка.iloc[:, 0] / сверка.iloc[:, 1] - 1) * 100
print(f"L_тΣ = {L_тΣ/1e3:.1f} кДж/кг, H_т = {H_т_ст/1e3:.1f} кДж/кг, η_т = {η_т_турб:.3f}, η_л = {η_л_турб:.3f}, "
      f"H*_т = {H_т_полн/1e3:.1f} кДж/кг, η*_т = {η_т_полн:.3f}")
print(f"N_т = {N_т/1e3:.1f} кВт (η_м = {η_м:.4f}), N_к = {N_к/1e3:.1f} кВт, "
      f"на валу N_в = {N_вал/1e3:.1f} кВт (задано {Ц['N_зад']/1e3:.0f} кВт)")
сверка.style.format("{:.4g}")

## 9. Меридиональное сечение и угол раскрытия (п. 64–69, стр. 55, 61–62)
Угол раскрытия γ = γ_н + γ_в должен быть 0…20° (стр. 55), при γ > 25° возможен отрыв потока. Желательное значение — 12°.
По книге (стр. 61) нужный γ получают изменением ширины венцов b_a и осевых зазоров: если на участке γ > 12°,
венец расширяется (до l/b_a = 1,0 у СА и 1,5 у РК). Если и тогда γ > 25° — нужно больше ступеней.


In [ ]:
def γ_участка(r_н1, r_в1, r_н2, r_в2, dx):
    """Угол раскрытия γ = γ_н + γ_в на участке длиной dx, град."""
    return math.degrees(math.atan((r_н2 - r_н1) / dx) + math.atan((r_в1 - r_в2) / dx))


def ширина(l, lb, lb_мин, δ_до, r1, r2, коэф_δ):
    """Ширина венца b_a: исходная l/(l/b_a); если γ на участке > γ_цель - расширяем венец (стр. 61), но не шире l/(l/b_a)_мин.

    r1, r2 = (r_н, r_в) в начале и в конце участка; длина участка dx = δ_до + коэф_δ·b + b."""
    b0 = max(l / lb, b_a_мин)
    b_макс = max(l / lb_мин, b0)
    γ_от = lambda b: γ_участка(*r1, *r2, δ_до + (1 + коэф_δ) * b)
    if γ_от(b0) <= γ_цель:
        return b0
    if γ_от(b_макс) > γ_цель:
        return b_макс
    lo, hi = b0, b_макс
    for _ in range(50):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if γ_от(mid) > γ_цель else (lo, mid)
    return hi


def r_нв(D, l):
    return (D / 2 + l / 2, D / 2 - l / 2)


контур = []  # сечения по ходу газа: x, r_наружный, r_внутренний
геом = []
x = 0.0
s0 = ступени[0]
b_c = max(s0["l_c"] / lb_c, b_a_мин)  # СА 1-й ступени: перед ним раскрытия нет
r_пред = r_нв(s0["D_c"], s0["l_c"])
контур.append((x, *r_пред, "СА1 выход"))
for i, s in enumerate(ступени):
    # участок СА(i) выход -> РК(i) выход: dx = δ_a(СА) + b_л
    δ_a = δа_к_ba * b_c
    r_л = r_нв(s["D_л"], s["l_л"])
    b_л = ширина(s["l_л"], lb_л, lb_мин_л, δ_a, r_пред, r_л, 0.0)
    x += δ_a + b_л
    контур.append((x, *r_л, f"РК{i+1} выход"))
    геом.append(dict(b_c=b_c, b_л=b_л, δ_a=δ_a, δ_a_л=δа_к_ba * b_л))
    r_пред = r_л
    if i + 1 < len(ступени):
        # участок РК(i) выход -> СА(i+1) выход: dx = δ_a(РК) + b_c
        сл = ступени[i + 1]
        r_c = r_нв(сл["D_c"], сл["l_c"])
        δ_a_л = δа_к_ba * b_л
        b_c = ширина(сл["l_c"], lb_c, lb_мин_c, δ_a_л, r_пред, r_c, 0.0)
        # переходный канал при смене диаметра: наклон стенок не круче γ_макс
        x += max(δ_a_л + b_c, max(abs(r_c[0] - r_пред[0]), abs(r_c[1] - r_пред[1])) / math.tan(math.radians(γ_макс)))
        контур.append((x, *r_c, f"СА{i+2} выход"))
        r_пред = r_c
контур_df = pd.DataFrame(контур, columns=["x", "r_н", "r_в", "сечение"])
γ_н = np.degrees(np.arctan(np.diff(контур_df.r_н) / np.diff(контур_df.x)))
γ_в = np.degrees(np.arctan(-np.diff(контур_df.r_в) / np.diff(контур_df.x)))
участки = pd.DataFrame(
    {"участок": [f"{контур_df.сечение[j]} → {контур_df.сечение[j+1]}" for j in range(len(γ_н))],
     "γ_н, град": γ_н, "γ_в, град": γ_в, "γ = γ_н+γ_в, град": γ_н + γ_в}
)
# угол раскрытия турбины в целом: от выхода СА 1-й ступени до выхода РК последней
dx_т = контур_df.x.iloc[-1] - контур_df.x.iloc[0]
γ_турб = γ_участка(контур_df.r_н.iloc[0], контур_df.r_в.iloc[0], контур_df.r_н.iloc[-1], контур_df.r_в.iloc[-1], dx_т)
γ_max = участки["γ = γ_н+γ_в, град"].max()
ширины = pd.DataFrame(
    {"b_a СА, мм": [g["b_c"] * 1e3 for g in геом], "l/b_a СА": [s["l_c"] / g["b_c"] for s, g in zip(ступени, геом)],
     "b_a РК, мм": [g["b_л"] * 1e3 for g in геом], "l/b_a РК": [s["l_л"] / g["b_л"] for s, g in zip(ступени, геом)]},
    index=[f"ст. {i}" for i in range(1, len(ступени) + 1)],
)
print(f"Длина проточной части {dx_т*1e3:.0f} мм; угол раскрытия турбины в целом γ = {γ_турб:.1f}°, "
      f"на участках - до {γ_max:.1f}° (цель {γ_цель:.0f}°, не более {γ_макс:.0f}°)")
if γ_max > γ_макс:
    print(f"ВНИМАНИЕ: угол раскрытия {γ_max:.1f}° > {γ_макс:.0f}° - увеличьте число ступеней z")
elif γ_max > γ_цель + 0.05:
    print(f"угол раскрытия {γ_max:.1f}° больше желательного {γ_цель:.0f}° даже при l/b_a = {lb_мин_c}/{lb_мин_л} (допустимо до {γ_макс:.0f}°)")
print(ширины.round(2).to_string())
участки.style.format({"γ_н, град": "{:.1f}", "γ_в, град": "{:.1f}", "γ = γ_н+γ_в, град": "{:.1f}"})


## 10. Профилирование лопаток по радиусу (гл. 7)
Для каждой ступени — корневое, среднее и периферийное сечения на выходе из СА и из РК.
* **Постоянная циркуляция** (Γ₁ᵤ = const, формулы 67–82): c₁ᵤ·r = const, c₁ₐ = const,
  давление по (71), реактивность по (72)–(73), углы по (75)–(82).
* **Постоянный угол α₁** (разд. 7.3): c ∝ r^(−cos²α₁).
  Для выхода из РК книга формул не даёт — принято L_u = const по радиусу (допущение): c₂ᵤ = (L_u − c₁ᵤu₁)/u₂.

**Расчётный закон — Γ = const** (`закон_профилирования` в ячейке с допущениями); закон α₁ = const показан для сравнения.
У длинных лопаток α₁ ср повышается ступенью так, чтобы у корня α₁ ≥ `α1_мин_корня` (tg α₁кор = tg α₁ср · r_кор / r_ср).


In [ ]:
def профиль(s, метод):
    """Параметры на радиусах корня / среднем / периферии для одной ступени."""
    k, cp, T0, p0, H0, p2, n = s["k"], s["cp"], s["T0"], s["p0"], s["H0"], s["p2"], n_т
    r_c, r_л = s["D_c"] / 2, s["D_л"] / 2
    out = {}
    for имя, f_c, f_л in (("корень", -0.5, -0.5), ("средний", 0.0, 0.0), ("периферия", 0.5, 0.5)):
        r1 = r_c + f_c * s["l_c"]
        r2 = r_л + f_л * s["l_л"]
        u1 = math.pi * n * r1 / 30
        u2 = math.pi * n * r2 / 30
        if метод == "Γ":
            c1u = s["c1u"] * r_c / r1
            c1a = s["c1a"]
            a1 = math.sqrt(k * R * s["T1_ад"])
            M1u = s["c1u"] / a1
            p1 = s["p1"] * (1 + (k - 1) / 2 * M1u**2 * (1 - (r_c / r1) ** 2)) ** (k / (k - 1))  # (71)
            c2u = s["c2u"] * r_л / r2
            c2a = s["c2a"]
        else:  # α1 = const
            m = math.cos(math.radians(s["α1"])) ** 2
            q = (r_c / r1) ** m
            c1u, c1a = s["c1u"] * q, s["c1a"] * q
            c1ад = math.hypot(c1u, c1a) / φ
            p1 = p0 * (1 - c1ад**2 / (2 * cp * T0)) ** (k / (k - 1))
            c2a = s["c2a"]
            c2u = (s["L_u"] - c1u * u1) / u2 if u2 else s["c2u"]
        T1ад = T0 * (p1 / p0) ** ((k - 1) / k)
        ρ_r = cp * T1ад / H0 * (1 - (p2 / p1) ** ((k - 1) / k))  # (72)-(73)
        out[имя] = {
            "r, мм": r1 * 1e3,
            "c₁, м/с": math.hypot(c1u, c1a),
            "p₁/p*₀": p1 / p0,
            "ρ": ρ_r,
            "α₁, град": math.degrees(math.atan2(c1a, c1u)),
            "β₁, град": math.degrees(math.atan2(c1a, c1u - u1)),
            "α₂, град": math.degrees(math.atan2(c2a, c2u)),
            "β₂, град": math.degrees(math.atan2(c2a, c2u + u2)),
        }
    return pd.DataFrame(out)


профили = {м: [профиль(s, м) for s in ступени] for м in ("Γ", "α1")}
пред = []
законы = закон_профилирования if isinstance(закон_профилирования, (list, tuple)) else [закон_профилирования] * len(ступени)
список = [профили[з][j] for j, з in enumerate(законы)]  # предупреждения - только по расчётному закону ступени; второй закон справочный
if True:
    for i, (df, м) in enumerate(zip(список, законы), 1):
        if df.loc["α₁, град", "корень"] < α1_мин_корня:
            пред.append(f"{м}, ст. {i}: α₁ у корня = {df.loc['α₁, град', 'корень']:.1f}° < {α1_мин_корня:.0f}° (стр. 73)")
        if df.loc["ρ", "корень"] < ρ_корня_мин:
            пред.append(f"{м}, ст. {i}: ρ у корня = {df.loc['ρ', 'корень']:.3f} < {ρ_корня_мин}")
print("Предупреждения профилирования:" if пред else "Критерии профилирования выполнены.")
print("\n".join("  - " + t for t in пред))
сводка_профиль = pd.concat(
    {f"ст. {i} ({м})": df.loc[["α₁, град", "β₁, град", "ρ", "α₂, град", "β₂, град"]] for м in ("Γ", "α1") for i, df in enumerate(профили[м], 1)}
)
сводка_профиль.style.format("{:.3g}")

## 11. Решётки профилей: шаг, хорда, число лопаток (гл. 8, аналитическая часть)
Построение самого контура профиля (графическое) здесь не выполняется. Считаем:
углы лопаток с учётом отставания потока δ, угол установки γ, оптимальный относительный шаг,
хорду, шаг, число лопаток (в соседних решётках — взаимно простые), горло a ≈ t·sinα₁.

In [ ]:
def отставание(M, угол):
    """δ, град (стр. 78): зависит от числа Маха и угла выхода."""
    δ06 = np.interp(угол, [17.5, 25.0, 35.0], [0.9, 1.5, 3.5])  # при M = 0,6
    δ_95 = 1 / 3  # 20' при M >= 0,95
    return float(np.interp(M, [0.6, 0.95], [δ06, δ_95]))


def t_к_b(β1, β2, K, c_max):
    s = β1 + β2
    if s >= 180 or β2 <= 0:
        return 0.75
    t_b = K * (180 / (180 - s) * math.sin(math.radians(β1)) / math.sin(math.radians(β2))) ** (1 / 3) * (1 - c_max)
    return float(np.clip(t_b, 0.55, 0.9))


def γ_установки(β1, β2):
    d = β1 - β2
    return 70 - 0.127 * d - 0.0041 * d**2


def взаимно_простые(z_c, t_л, D):
    """Число лопаток РК, ближайшее к πD/t и взаимно простое с z_c."""
    z0 = round(math.pi * D / t_л)
    for d in (0, 1, -1, 2, -2, 3, -3):
        if math.gcd(z_c, z0 + d) == 1:
            return z0 + d
    return z0


решётки = []
for i, (s, g) in enumerate(zip(ступени, геом), 1):
    охл = i <= охл_ступеней
    D = 0.5 * (s["D_c"] + s["D_л"])
    # --- СА ---
    a1 = math.sqrt(s["k"] * R * s["T1"])
    M_c1 = s["c1"] / a1
    δ_c = отставание(M_c1, s["α1"])
    α1_л = s["α1"] - δ_c
    β_вх_c = 90.0 if i == 1 else ступени[i - 2]["α2"]
    γ_c = γ_установки(β_вх_c, α1_л)
    b_c = g["b_c"] / math.sin(math.radians(γ_c))
    tb_c = t_к_b(β_вх_c, α1_л, 0.45, 0.2 if охл else 0.13)
    z_c = max(3, round(math.pi * D / (tb_c * b_c)))
    t_c = math.pi * D / z_c
    a_c = t_c * math.sin(math.radians(α1_л))
    s2_c = (0.11 if охл else 0.055) * a_c
    # --- РК ---
    w_зв = math.sqrt(s["k"] * R * s["T2"])
    M_w2 = s["w2"] / w_зв
    δ_л = отставание(M_w2, s["β2"])
    β2_л = s["β2"] - δ_л
    γ_л = γ_установки(s["β1"], β2_л)
    b_л = g["b_л"] / math.sin(math.radians(γ_л))
    tb_л = t_к_b(s["β1"], β2_л, 0.60, 0.2 if охл else 0.11)
    z_л = взаимно_простые(z_c, tb_л * b_л, D)
    t_л = math.pi * D / z_л
    a_л = t_л * math.sin(math.radians(β2_л))
    s2_л = (0.17 if охл else 0.07) * a_л
    решётки.append(
        {
            "M_c1": M_c1, "δ_c, град": δ_c, "α1л, град": α1_л, "γ_c, град": γ_c, "b_c, мм": b_c * 1e3,
            "t/b_c": tb_c, "z_c": z_c, "t_c, мм": t_c * 1e3, "a_c, мм": a_c * 1e3, "s2_c, мм": s2_c * 1e3,
            "r1_c, мм": 0.035 * g["b_c"] * 1e3, "l/b_c (по хорде)": s["l_c"] / b_c,
            "M_w2": M_w2, "δ_л, град": δ_л, "β2л, град": β2_л, "γ_л, град": γ_л, "b_л, мм": b_л * 1e3,
            "t/b_л": tb_л, "z_л": z_л, "t_л, мм": t_л * 1e3, "a_л, мм": a_л * 1e3, "s2_л, мм": s2_л * 1e3,
            "r1_л, мм": 0.035 * g["b_л"] * 1e3, "l/b_л (по хорде)": s["l_л"] / b_л,
        }
    )  # fmt: skip
решётки_df = pd.DataFrame(решётки, index=[f"ст. {i}" for i in range(1, len(ступени) + 1)]).T
решётки_df.style.format("{:.3g}")

## 12. Контроль расчёта
Жёсткие проверки (ошибка = неверный расчёт) и мягкие замечания (проектное решение пользователя).

In [ ]:
# --- жёсткие проверки ---
for i, s in enumerate(ступени, 1):
    assert abs(s["η_u"] - s["η_u2"]) < 1e-6, f"ст. {i}: η_u по п. 102 и п. 111 не совпали"
    T2_полн = s["T2"] + s["c2"] ** 2 / (2 * s["cp"])
    assert abs(s["L_u"] - s["cp"] * (s["T0"] - T2_полн)) < 1e-6 * s["L_u"], f"ст. {i}: нарушен баланс энергии"
assert abs(ступени[-1]["p2"] - p_кон) < 1e-6 * p_кон, "давление за последней ступенью ≠ p_т"
assert abs(N_вал / N_зад - 1) < 1e-3, "мощность на валу не равна заданной"
assert abs(A_факт - A_т) / A_т < 1e-3, "итерация по A_т не сошлась"
if n_н is None:
    assert σ_л_факт <= σ_л_доп * 1.005, "σ_л превышает допустимое"
for i, s in enumerate(ступени, 1):
    assert α2_диап[0] - 0.5 <= s["α2"] <= α2_диап[1] + 0.5, f"ст. {i}: α2 = {s['α2']:.1f}° вне допуска {α2_диап}"
    assert s["π_ст"] <= π_ст_макс, f"ст. {i}: π = {s['π_ст']:.2f} > {π_ст_макс} - запирание"
assert ступени[0]["l_c"] >= l_c1_мин, f"l_c 1-й ступени {ступени[0]['l_c']*1e3:.1f} мм < {l_c1_мин*1e3:.0f} мм"
assert γ_max <= γ_макс, f"угол раскрытия {γ_max:.1f}° > {γ_макс}° - нужно больше ступеней"
print("Жёсткие проверки пройдены: η_u (п. 102 = п. 111), баланс энергии каждой ступени, p₂z = p_т, N_вал = N_зад, σ_л, α₂ = 90°, "
      f"π ступеней <= {π_ст_макс}, угол раскрытия <= {γ_макс:.0f}°, l_c1 = {ступени[0]['l_c']*1e3:.1f} мм >= {l_c1_мин*1e3:.0f} мм.")

# --- мягкие замечания ---
зам = []
нет = lambda усл: [i for i, s in enumerate(ступени, 1) if усл(s)]  # номера ступеней по условию
if нет(lambda s: s["α1"] < α1_мин - 0.01):
    зам.append(f"α₁ < {α1_мин}° в ступенях {нет(lambda s: s['α1'] < α1_мин - 0.01)}")
if нет(lambda s: s["l_c"] < 40e-3):
    зам.append(
        f"короткие лопатки (l_c < 40...50 мм) в ступенях {нет(lambda s: s['l_c'] < 40e-3)}: "
        f"l_c = {ступени[0]['l_c']*1e3:.1f}...{ступени[-1]['l_c']*1e3:.1f} мм — концевые потери выше; φ и ψ по длине лопатки не корректировались"
    )
if нет(lambda s: s["ξ_з"] > 0.05):
    зам.append(
        f"потери в радиальном зазоре ξ_з > 5 % в ступенях {нет(lambda s: s['ξ_з'] > 0.05)}: "
        f"ξ_з(1-я) = {ступени[0]['ξ_з']:.3f} при δ_л = {ступени[0]['δ_л']*1e3:.2f} мм, l_л = {ступени[0]['l_л']*1e3:.1f} мм (бандаж/соты не учтены)"
    )
if u_т > 400:
    зам.append(f"u_т = {u_т:.0f} м/с > 400 м/с")
if γ_max > γ_цель + 0.05:
    зам.append(f"угол раскрытия на участках до {γ_max:.1f}° > желательных {γ_цель:.0f}° (допустимо до {γ_макс:.0f}°)")
if нет(lambda s: not π_ст_цель[0] - 0.05 <= s["π_ст"] <= π_ст_цель[1] + 0.05):
    зам.append("π ступеней " + ", ".join(f"{s['π_ст']:.2f}" for s in ступени) + f" - вне желательных {π_ст_цель[0]}...{π_ст_цель[1]} (запирание при {π_ст_макс})")
if η_т_полн < 0.90:
    зам.append(f"η*_т = {η_т_полн:.3f} < 0,90 (цикл: {η_т_полн_цикл:.3f})")
if abs(N_вал / Ц["N_зад"] - 1) > 0.02:
    зам.append(f"мощность на валу {N_вал/1e3:.1f} кВт вместо заданной {Ц['N_зад']/1e3:.0f} кВт ({(N_вал/Ц['N_зад']-1)*100:+.1f}%)")
if abs(G_в / G_в_цикл - 1) > 0.05:
    зам.append(f"расход воздуха для заданной мощности на валу {G_в:.3f} кг/с вместо {G_в_цикл:.3f} кг/с цикла ({(G_в/G_в_цикл-1)*100:+.1f}%): η*т цикла = {η_т_полн_цикл:.3f}, по книге {η_т_полн:.3f}")
if abs(ступени[-1]["c2"] / c_т_а_цикл - 1) > 0.05:
    зам.append(f"скорость за турбиной {ступени[-1]['c2']:.0f} м/с вместо принятых в цикле {c_т_а_цикл:.0f} м/с (M_т)")
print("Замечания:" if зам else "Замечаний нет.")
print("\n".join("  - " + t for t in зам))

## 13. Сводка и графики

In [ ]:
сводка_итог = pd.Series(
    {
        "Число ступеней z": z,
        "Длина сопловой лопатки 1-й ступени l_c1, мм": ступени[0]["l_c"] * 1e3,
        "z по формуле п. 54": z_расч,
        **{f"u/c0 ступени {i}": u_c0(s) for i, s in enumerate(ступени, 1)},
        **{f"π = p*₀/p₂ ступени {i}": s["π_ст"] for i, s in enumerate(ступени, 1)},
        "Частота вращения n_т, об/мин": n_т,
        "Средний диаметр D_т, мм": D_т * 1e3,
        "Окружная скорость u_т, м/с": u_т,
        "Напряжение в лопатках σ_л, МПа": σ_л_факт,
        "Удельная работа L_тΣ, кДж/кг": L_тΣ / 1e3,
        "Мощностной КПД η_т": η_т_турб,
        "Лопаточный КПД η_л": η_л_турб,
        "КПД по параметрам торможения η*_т": η_т_полн,
        "Мощность турбины N_т, кВт": N_т / 1e3,
        "Скорость за турбиной c₂, м/с": ступени[-1]["c2"],
        "Длина проточной части, мм": контур_df.x.iloc[-1] * 1e3,
        "Угол раскрытия турбины γ, град": γ_турб,
        "Угол раскрытия max на участке, град": γ_max,
        "Отбор на охлаждение дисков g_охл, %": g_охл * 100,
    }
)
сводка_итог.to_frame("значение").style.format("{:.4g}")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
# 1. меридиональное сечение
a = ax[0, 0]
a.plot(контур_df.x * 1e3, контур_df.r_н * 1e3, "k-o", ms=3)
a.plot(контур_df.x * 1e3, контур_df.r_в * 1e3, "k-o", ms=3)
a.axhline(D_т / 2 * 1e3, color="gray", ls="--", lw=0.8)  # средний радиус
a.set(xlabel="x, мм", ylabel="r, мм", title="Меридиональное сечение проточной части")
a.set_aspect("equal", adjustable="datalim")  # без искажения пропорций
a.grid(alpha=0.3)
# 2. давление и температура по ступеням
a = ax[0, 1]
ид = np.arange(1, z + 1)
л_p, = a.plot(ид, [s["p2"] / 1e3 for s in ступени], "o-", color="tab:blue", label="p₂, кПа")
a.set(xlabel="ступень", ylabel="p₂, кПа", title="Давление и КПД по ступеням")
a.set_yscale("log")
a2 = a.twinx()
л_1, = a2.plot(ид, [s["η_т_п"] for s in ступени], "s-", color="tab:red", label="η*_т")
л_2, = a2.plot(ид, [s["η_т"] for s in ступени], "^-", color="tab:green", label="η_т")
a2.set_ylabel("КПД")
a.legend(handles=[л_p, л_1, л_2], loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, fontsize=8)
a.grid(alpha=0.3)
# 3. треугольники скоростей (первая и последняя ступень)
a = ax[1, 0]
точки_тр = [(0.0, 0.0)]
for j, (idx, цвет) in enumerate(((0, "tab:blue"), (len(ступени) - 1, "tab:red"))):
    s = ступени[idx]
    for (vx, vy, лаб, ls) in (
        (s["c1u"], s["c1a"], "c₁", "-"),
        (s["c1u"] - s["u1"], s["c1a"], "w₁", "--"),
    ):
        a.annotate("", xy=(vx, vy), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=цвет, ls=ls))
        a.text(vx, vy, f"{лаб} ст.{idx+1}", color=цвет, fontsize=8)
        точки_тр.append((vx, vy))
    for (vx, vy, лаб, ls) in (
        (s["c2u"], -s["c2a"], "c₂", "-"),
        (s["c2u"] + s["u2"], -s["c2a"], "w₂", "--"),
    ):
        a.annotate("", xy=(vx, vy), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=цвет, ls=ls))
        a.text(vx, vy, f"{лаб} ст.{idx+1}", color=цвет, fontsize=8)
        точки_тр.append((vx, vy))
a.plot(*zip(*точки_тр), ".", alpha=0)  # только для автомасштаба (annotate его не двигает)
a.margins(0.15)
a.set(xlabel="окружная составляющая, м/с", ylabel="осевая составляющая, м/с", title="Треугольники скоростей (1-я и последняя ступени)")
a.axhline(0, color="gray", lw=0.5)
a.grid(alpha=0.3)
# 4. углы по радиусу в последней ступени
a = ax[1, 1]
for м, ls in (("Γ", "-"), ("α1", "--")):
    df = профили[м][-1]
    for пар, цвет in (("α₁, град", "tab:blue"), ("β₁, град", "tab:orange"), ("β₂, град", "tab:green")):
        a.plot(df.loc["r, мм"], df.loc[пар], ls, color=цвет, marker="o", label=f"{пар} ({'Γ=const' if м=='Γ' else 'α₁=const'})")
a.set(xlabel="r, мм", ylabel="угол, град", title=f"Профилирование по радиусу, ступень {z}")
a.legend(fontsize=7)
a.grid(alpha=0.3)
fig.tight_layout()
plt.show()